# Synthetic Time Series Moving Average Demo
Evaluating 3-point moving average against naive last-value forecasting on synthetic Ornstein-Uhlenbeck and sine wave time series.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

In [ ]:
import json
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-4b74fb-self-normalized-phase-space-adaptive-mov/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data = load_data()
print(f"Loaded {len(data['datasets'])} dataset groups.")

## Configuration
Set tunable parameters for the moving average evaluation.

In [ ]:
# Configuration parameters
WINDOW_SIZE = 3
GROUP_INDEX = 0
EXAMPLE_INDEX = 0

## Moving Average vs Naive Forecast Evaluation
Extract a synthetic time series sequence, compute a 3-point moving average, and compare its Mean Squared Error (MSE) against a naive last-value baseline.

In [ ]:
# Extract example series
group = data['datasets'][GROUP_INDEX]
example = group['examples'][EXAMPLE_INDEX]

noisy_series = np.array(json.loads(example['input']))
clean_trajectory = np.array(json.loads(example['output']))

# 3-point moving average
def moving_average(series, window):
    weights = np.repeat(1.0, window) / window
    return np.convolve(series, weights, 'valid')

ma_series = moving_average(noisy_series, WINDOW_SIZE)
t_indices = np.arange(WINDOW_SIZE - 1, len(noisy_series))
naive_vals = noisy_series[t_indices - 1]
ma_vals = ma_series[:len(t_indices)]
ground_truth = clean_trajectory[t_indices]

# Compute MSE
mse_ma = np.mean((ma_vals - ground_truth) ** 2)
mse_naive = np.mean((naive_vals - ground_truth) ** 2)

print(f"Dataset Group: {group['dataset']}")
print(f"Process Type: {example['metadata_process_type']}")
print(f"Noise Level: {example['metadata_noise_level']}")
print(f"3-point Moving Average MSE: {mse_ma:.4f}")
print(f"Naive Last-Value Forecast MSE: {mse_naive:.4f}")
print(f"Moving Average beats Naive? {mse_ma < mse_naive}")

## Visualization & Summary
Plotting the noisy input, ground truth clean trajectory, 3-point moving average, and naive forecast.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(noisy_series, label='Noisy Input', color='lightgray', alpha=0.8)
plt.plot(clean_trajectory, label='Ground Truth Clean', color='black', linestyle='--', linewidth=2)
plt.plot(t_indices, ma_vals, label=f'{WINDOW_SIZE}-pt Moving Average', color='blue', linewidth=2)
plt.plot(t_indices, naive_vals, label='Naive Last-Value', color='orange', linestyle=':', linewidth=2)

plt.title(f"Smoothing Comparison ({group['dataset']} - {example['metadata_process_type']})")
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()